# Inverse problems: evidence and non-uniqueness

**Level 1 · Prerequisites: none; reading time: 30 minutes.**

## Objectives

For model m, observations d, forward map F and error ε, write d=F(m)+ε. A good fit constrains predictions on the acquisition geometry; it does not prove a unique or geologically correct structure. This lesson distinguishes artifact identity, numerical validation and field interpretation.

## Setup

Use the repository's reviewed Python 3.12 notebook environment. This notebook installs nothing. The selected product artifact is synthetic, not field data.

The next cell verifies the committed result bytes and reads its receipt; it does not rerun a solver.

In [ ]:
import hashlib, json
from pathlib import Path

# Resolve product data from the repository root; the notebook runner starts in this lesson's folder.
root = Path.cwd().parents[2]
case = root / 'data/derived/m01-scientific-course/prism-case-0'
receipt = json.loads((case / 'receipt.json').read_text(encoding='utf-8'))
digest = hashlib.sha256((case / 'result.json').read_bytes()).hexdigest()
print({'case': 'synthetic prism control', 'result_sha256': digest, 'receipt_fields': sorted(receipt)})

## Data

The request, result and receipt are authored synthetic-control artifacts. A hash proves byte identity only; it cannot establish physical correctness, convergence, uniqueness or field eligibility. No field survey is represented.

**Files:** `data/derived/m01-scientific-course/prism-case-0/request.json`, `result.json`, `receipt.json`.

## Method

A common objective is φ(m)=½‖W_d(F(m)−d)‖²+βφ_m(m−m_ref). W_d represents declared data uncertainty; β and φ_m encode prior assumptions. Different meshes, bounds, references and acquisition geometries can fit the same d with different models. SimPEG is a framework for simulation and inversion ([Cockett et al., 2015](https://doi.org/10.1016/j.cageo.2015.09.015)); a reproducible artifact still requires independent numerical controls before a physical claim.

A useful local view is the Jacobian J=∂F/∂m. A perturbation δm produces δd≈Jδm; directions in the null space of J are invisible to this experiment, while small singular values describe weakly constrained combinations. Thus “the prediction matches” does not imply “the geology is recovered.” A regularized estimate minimizes a data term plus a prior, for example φ=½‖W_d(F(m)−d)‖²+β‖L(m−m_ref)‖². W_d scales residuals by declared errors and L penalizes selected model variations. If d and σ are in mGal, W_d r is dimensionless. Changing β, the mesh or L can change the selected model even when its data misfit barely changes. SimPEG frames this distinction as geophysical simulation and parameter estimation [Cockett et al. (2015)](https://doi.org/10.1016/j.cageo.2015.09.015).

The persisted product artifact is an authored synthetic gravity case, not a field survey. Its residual map reveals spatial support but is neither independent validation nor a subsurface image; it inherits fixed geometry and noise assumptions. A checksum verifies byte identity, while numerical validity requires recomputation and independent controls. The figure distinguishes training from outer holdout and masks unsupported predictions.

The Jacobian J=∂F/∂m maps a small model change to data change, δd≈Jδm. Null-space directions are invisible to this acquisition; small singular values indicate weakly constrained combinations. Thus a matching prediction does not establish recovered geology. A regularized estimate may minimize φ=½‖W_d(F(m)−d)‖²+β‖L(m−m_ref)‖². W_d scales residuals by stated errors, while L and β encode a model preference. With d,σ in mGal, W_dr is dimensionless. Changing the mesh or regularizer can select another model with similar fit. SimPEG presents this simulation/parameter-estimation framing [Cockett et al. (2015)](https://doi.org/10.1016/j.cageo.2015.09.015).

The stored artifact is an authored synthetic gravity case, not a field survey. Its residual map reveals spatial support but is neither independent validation nor a subsurface image; it inherits fixed geometry and noise assumptions. A checksum verifies byte identity, while numerical validity requires recomputation and independent controls. The figure separates training from outer holdout and masks unsupported predictions.

## Results

The code verifies artifact identity only. It does not recompute a forward model or solve an inverse problem. The stored control is synthetic and says nothing about field geology.

Use the stored product residuals at their actual local east/north coordinates. Color is prediction minus observation in mGal; hollow squares mark outer holdout. This is a diagnostic of a synthetic case, not an observed field map.


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Plot persisted residuals and compute supported split RMSE; these values belong to an authored synthetic case.
case = Path.cwd().parents[2] / 'data/derived/m01-scientific-course/prism-case-0'
a = json.loads((case / 'result.json').read_text(encoding='utf-8'))
xy = np.column_stack([a['geometry']['easting_m'], a['geometry']['northing_m']])
r = np.asarray(a['stations']['signed_predicted_minus_observed_mgal'], dtype=float)
covered = np.asarray(a['stations']['prediction_covered'], dtype=bool)
part = np.asarray(a['split']['partition'])
fig, ax = plt.subplots(figsize=(8.2, 5.6), constrained_layout=True)
sc = ax.scatter(xy[covered,0], xy[covered,1], c=r[covered], cmap='coolwarm', s=34, edgecolors='0.25', linewidths=.3)
hold = covered & (part == 'holdout')
ax.scatter(xy[hold,0], xy[hold,1], facecolors='none', edgecolors='black', marker='s', s=70, label='Outer holdout')
ax.set(xlabel='Easting [m]', ylabel='Northing [m]', title='Synthetic case: supported station residuals')
ax.set_aspect('equal', adjustable='datalim'); ax.legend()
fig.colorbar(sc, ax=ax, label='Predicted − observed $g_z$ [mGal]')
for split in ('train','holdout'):
    use = covered & (part == split)
    print(f'{split}: n={use.sum()}, RMSE={np.sqrt(np.mean(r[use]**2)):.5f} mGal')
plt.show()


## Summary

Data fit, model recovery and geological interpretation are separate claims. State the physics, geometry, errors, prior and discretization that condition every inverse result.


## References

- https://doi.org/10.1016/j.cageo.2015.09.015